# 🌍 Nettoyage complet des données - Géolocalisation

Ce notebook contient le script intégral pour nettoyer les données géographiques (`olist_geolocation_dataset.csv`), incluant l'analyse des outliers IQR.


## 1. Configuration des chemins et du fichier

Configuration de l'environnement de données.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import os

RAW_DIR = Path(r"C:\Users\malek\OneDrive\Desktop\ESPRIT\3ème IA\PROJET BI\data\raw")
PROCESSED_DIR = RAW_DIR.parent / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FILE_NAME = "olist_geolocation_dataset.csv"
input_path = RAW_DIR / FILE_NAME
output_path = PROCESSED_DIR / f"{input_path.stem}_cleaned.csv"


## 2. Vérification et lecture du fichier

Lecture du dataset de géolocalisation.


In [ ]:
if not input_path.exists():
    raise FileNotFoundError(f"Fichier introuvable : {input_path}")

df = pd.read_csv(input_path, encoding="utf-8-sig")

print("Dimensions initiales :", df.shape)
print("\nColonnes initiales :")
print(df.columns.tolist())
print("\nTypes initiaux :")
print(df.dtypes)


## 3. Nettoyage des noms de colonnes

Normalisation des en-têtes.


In [ ]:
df.columns = df.columns.str.replace(r"[^\w]", "", regex=True)

print("\nColonnes après nettoyage des noms :")
print(df.columns.tolist())


## 4. Nettoyage de la colonne textuelle sélectionnée

Sanitisation de la variable `geolocation_state`.


In [ ]:
colonnes_texte = ["geolocation_state"]

for col in colonnes_texte:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace("", pd.NA)
    df[col] = df[col].str.replace(r"[^\w]", "", regex=True)


## 5. Suppression de la colonne non conservée

Retrait du champ `geolocation_city`.


In [ ]:
columns_to_remove = ["geolocation_city"]

df = df.drop(columns=columns_to_remove)

print("\nColonnes restantes :")
print(df.columns.tolist())


## 6. Suppression des lignes/colonnes vides et des doublons

Traitement des lignes vides et suppression des doublons.


In [ ]:
df = df.dropna(axis=0, how="all")
df = df.dropna(axis=1, how="all")

duplicates = df.duplicated().sum()
df = df.drop_duplicates()


## 7. Détection des valeurs atypiques (outliers)

Calcul des bornes de l'Écart Interquartile (IQR) sur la latitude, la longitude et le code postal pour détecter les valeurs extrêmes.


In [ ]:
colonnes_numeriques = [
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_zip_code_prefix"
]

for col in colonnes_numeriques:
    serie = pd.to_numeric(df[col], errors="coerce").dropna()
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < limite_inf) | (serie > limite_sup)]

    print(f"\nColonne : {col}")
    print("Valeurs négatives :")
    print(df[df[col] < 0])
    print("Valeur minimale :", df[col].min())
    print("Valeur maximale :", df[col].max())
    print("Limite inférieure :", limite_inf)
    print("Limite supérieure :", limite_sup)
    print("Nombre d'outliers :", len(outliers))
    print("Pourcentage :", round(len(outliers) / len(serie) * 100, 2), "%")


## 8. Traitement des codes postaux

Remplacement des valeurs d'origine égale à 0 par `NaN` et cast du code postal en type `string`.


In [ ]:
df["geolocation_zip_code_prefix"] = df["geolocation_zip_code_prefix"].replace(0, np.nan)
df["geolocation_zip_code_prefix"] = df["geolocation_zip_code_prefix"].astype("string")

print("\nTypes après traitement des codes postaux :")
print(df.dtypes)


## 9. Bilan final du nettoyage

Affichage du rapport de nettoyage.


In [ ]:
print("\nNettoyage terminé.")
print("Doublons supprimés :", duplicates)
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())
print("\nDimensions finales :", df.shape)


## 10. Export du fichier nettoyé

Exportation CSV final.


In [ ]:
df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="NULL")

print("\nCopie enregistrée :", output_path)
print("Le fichier original reste intact.")
